# Self-Pruning Neural Network: analysis
Loads the results written by `python main.py --run_all` and the trained checkpoints, then looks at
how λ trades accuracy for sparsity and where in the network the pruning happens.

In [ ]:
import json, os
import pandas as pd
import torch
from IPython.display import Image, display

if not os.path.exists('results'):
    os.chdir('..')
from src.train import load_model, load_cifar10, Trainer, TrainConfig

results = json.load(open('results/results.json'))
pd.DataFrame([{k: r[k] for k in ('lambda', 'accuracy_soft', 'accuracy', 'sparsity', 'compression')} for r in results])

### 1. Trade-off
`accuracy_soft` is the model straight out of training; `accuracy` is after every gate below 0.01 is set to exactly 0.
If the two match, the pruned connections really were unnecessary.

In [ ]:
display(Image('results/lambda_comparison.png', width=650))

### 2. Gate distribution
A successful run shows a large spike at 0 and a separate cluster of surviving gates.

In [ ]:
for r in results:
    display(Image(f"results/gates_dist_lambda_{r['lambda']:g}.png", width=650))

### 3. Where does the pruning happen?
The 3072→512 input layer holds 90% of all weights and is pruned hardest: most raw pixels are redundant.

In [ ]:
display(Image('results/layer_sparsity.png', width=650))

### 4. Training dynamics

In [ ]:
for r in results:
    display(Image(f"results/curves_lambda_{r['lambda']:g}.png", width=900))

### 5. Which input pixels survive?
Summing the kept connections of `fc1` per input pixel shows which parts of the image the pruned network still looks at.

In [ ]:
import matplotlib.pyplot as plt
best = max(results, key=lambda r: r['sparsity'])
model, _ = load_model(f"results/model_lambda_{best['lambda']:g}.pth")
kept = model.net.fc1.mask.sum(0).view(3, 32, 32).sum(0)
plt.imshow(kept, cmap='magma'); plt.colorbar(label='connections kept'); plt.title(f"fc1 input usage, λ = {best['lambda']:g}"); plt.axis('off');